### Extract Load Transform
This notebook do the following transformation in **Dados Dados Carregamento_24-25-26_RAMPs.xlsx** and **Direção Vento_RAMPs 09 e 10.xlsx**.
- Converty datatype for each columns.
- Filter this data for **non-nan values from column Dados Dados Carregamento_24-25-26_RAMPs.xlsx**.
- Applying dummie encoding to **Produto**.
- Agregating columns **Wind Direction** from **Direção Vento_RAMPs 09 e 10.xlsx** using **Vector Averaging**.
- Getting Weather information.
- Applying MICE.

In [72]:
import pandas as pd 

#Getting data
df = pd.read_excel(r"/home/jorgemetri/Desktop/git_repositories/Fugitive-Industrial-Emission-Mining-Company/data/raw/Dados Carregamento_24-25-26_RAMPs.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19409 entries, 0 to 19408
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Dia / Hora                 19409 non-null  datetime64[us]
 1   Carregamento
(TMN)         19409 non-null  float64       
 2   Navio
Atracado             19409 non-null  int64         
 3   Produto                    7985 non-null   object        
 4    H20 (%)                   7050 non-null   float64       
 5   Finos 
-6,3mm (%)          5402 non-null   float64       
 6   Compressão
16,0(Kgf)       5402 non-null   float64       
 7   Compressão
12,5(Kgf)       5402 non-null   float64       
 8   Tempo Estoque
(dias)       4726 non-null   object        
 9   Descarga
Pet Coke          19097 non-null  float64       
 10  Descarga
Calcário          19097 non-null  float64       
 11  Taxa de Emissão PIER
kg/h  17205 non-null  float64       
 12  RAMP 09
3m (µg/

In [73]:
#Convert datatype for each column
for col in df.select_dtypes(include=["object"]).columns:
    # Replace whitespace-only strings with NaN
    cleaned = df[col].replace(r'^\s*$', pd.NA, regex=True)
    non_null = cleaned.dropna()

    if len(non_null) == 0:
        continue

    # Numeric if every non-null value is a number
    if pd.to_numeric(non_null, errors="coerce").notna().all():
        df[col] = pd.to_numeric(cleaned, errors="coerce")
        continue

    # Datetime if every non-null value is a date
    if pd.to_datetime(non_null, errors="coerce").notna().all():
        df[col] = pd.to_datetime(cleaned, errors="coerce")
#Renaming column Dia / Hora
df = df.rename(columns={'Dia / Hora':'Data-Hora'})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19409 entries, 0 to 19408
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Data-Hora                  19409 non-null  datetime64[us]
 1   Carregamento
(TMN)         19409 non-null  float64       
 2   Navio
Atracado             19409 non-null  int64         
 3   Produto                    7985 non-null   object        
 4    H20 (%)                   7050 non-null   float64       
 5   Finos 
-6,3mm (%)          5402 non-null   float64       
 6   Compressão
16,0(Kgf)       5402 non-null   float64       
 7   Compressão
12,5(Kgf)       5402 non-null   float64       
 8   Tempo Estoque
(dias)       4725 non-null   float64       
 9   Descarga
Pet Coke          19097 non-null  float64       
 10  Descarga
Calcário          19097 non-null  float64       
 11  Taxa de Emissão PIER
kg/h  17205 non-null  float64       
 12  RAMP 09
3m (µg/

/tmp/ipykernel_568577/2011687055.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  if pd.to_datetime(non_null, errors="coerce").notna().all():


In [74]:
df.columns

Index(['Data-Hora', 'Carregamento\n(TMN)', 'Navio\nAtracado', 'Produto',
       ' H20 (%)', 'Finos \n-6,3mm (%)', 'Compressão\n16,0(Kgf)',
       'Compressão\n12,5(Kgf)', 'Tempo Estoque\n(dias)', 'Descarga\nPet Coke',
       'Descarga\nCalcário', 'Taxa de Emissão PIER\nkg/h',
       'RAMP 09\n3m (µg/m³)', 'RAMP 09\n9m (µg/m³)', 'RAMP 09\n16m (µg/m³)',
       'RAMP 09\nDireção Vento [°]', 'RAMP 10\n3m (µg/m³)',
       'RAMP 10\n9m (µg/m³)', 'RAMP 10\n16m (µg/m³)',
       'RAMP 10\nDireção Vento [°]', 'RAMP 11\n3m (µg/m³)',
       'RAMP 11\n9m (µg/m³)', 'RAMP 11\n16m (µg/m³)', 'RAMP 12B\n9m (µg/m³)',
       'RAMP 12C\n9m (µg/m³)'],
      dtype='str')

In [75]:
#Filter dataframe by non-nan values from column "Taxa de Emissão PIER kg/h"
df= df.dropna(subset=['Taxa de Emissão PIER\nkg/h'])
df['Taxa de Emissão PIER\nkg/h']

1765     0.01
1766     0.03
1767     0.02
1768     0.04
1769     0.05
         ... 
19404    0.00
19405    0.00
19406    0.04
19407    0.11
19408    0.25
Name: Taxa de Emissão PIER\nkg/h, Length: 17205, dtype: float64

In [76]:
#Applying dummie encoding to Produto Column
df = pd.get_dummies(df,columns=['Produto'],dtype=int)
df

,Data-Hora,Carregamento\n(TMN),Navio\nAtracado,H20 (%),"Finos \n-6,3mm (%)","Compressão\n16,0(Kgf)","Compressão\n12,5(Kgf)",Tempo Estoque\n(dias),Descarga\nPet Coke,Descarga\nCalcário,...,Produto_PBF/STD,Produto_PDR-STD,Produto_PDR-STD - PBF-STD,Produto_PDR-STD / PBF-STD,Produto_PDR/STD,Produto_PF-SA,Produto_PFN,Produto_PFN/STD,Produto_PSC/STD,Produto_PSC/STD - PBF-MB45
1765,2024-03-14 12:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1766,2024-03-14 13:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1767,2024-03-14 14:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1768,2024-03-14 15:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1769,2024-03-14 16:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19404,2026-03-19 11:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
19405,2026-03-19 12:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
19406,2026-03-19 13:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
19407,2026-03-19 14:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [77]:
df['Data-Hora'].unique()

<DatetimeArray>
['2024-03-14 12:00:00', '2024-03-14 13:00:00', '2024-03-14 14:00:00',
 '2024-03-14 15:00:00', '2024-03-14 16:00:00', '2024-03-14 17:00:00',
 '2024-03-14 18:00:00', '2024-03-14 19:00:00', '2024-03-14 20:00:00',
 '2024-03-14 21:00:00',
 ...
 '2026-03-19 06:00:00', '2026-03-19 07:00:00', '2026-03-19 08:00:00',
 '2026-03-19 09:00:00', '2026-03-19 10:00:00', '2026-03-19 11:00:00',
 '2026-03-19 12:00:00', '2026-03-19 13:00:00', '2026-03-19 14:00:00',
 '2026-03-19 15:00:00']
Length: 17205, dtype: datetime64[us]

In [78]:
# Getting Weather information.
import pandas as pd
import requests
from typing import Sequence

OPENMETEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

DEFAULT_HOURLY_VARS = (
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "precipitation",
    "cloud_cover",
    "shortwave_radiation",
    "boundary_layer_height",
)


def fetch_openmeteo_hourly(
    df: pd.DataFrame,
    latitude: float,
    longitude: float,
    datetime_col: str = "Data-Hora",
    variables: Sequence[str] = DEFAULT_HOURLY_VARS,
    timezone: str = "America/Sao_Paulo",
    timeout: int = 60,
) -> pd.DataFrame:
    """
    Busca dados meteorológicos horários da Open-Meteo (ERA5) cobrindo todo
    o intervalo temporal presente em `df[datetime_col]`.

    Parâmetros
    ----------
    df : DataFrame com coluna datetime (ex.: 'Data-Hora', datetime64[us]).
    latitude, longitude : coordenadas do ponto.
    variables : lista de variáveis horárias (ver docs Open-Meteo).
    timezone : IANA tz. Retorna timestamps locais (tz-naive) alinhados a essa tz.

    Retorna
    -------
    DataFrame indexado por timestamp horário, uma coluna por variável.
    """
    times = pd.to_datetime(df[datetime_col])
    if times.dt.tz is not None:
        times = times.dt.tz_convert(timezone).dt.tz_localize(None)

    start_date = times.min().strftime("%Y-%m-%d")
    end_date   = times.max().strftime("%Y-%m-%d")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(variables),
        "timezone": timezone,
        "wind_speed_unit": "ms",   # m/s em vez do default km/h
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
    }

    resp = requests.get(OPENMETEO_ARCHIVE_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    payload = resp.json()

    hourly = payload["hourly"]
    weather = pd.DataFrame(hourly)
    weather["time"] = pd.to_datetime(weather["time"])
    weather = (
        weather
        .rename(columns={"time": datetime_col})
        .set_index(datetime_col)
        .sort_index()
    )

    return weather

import numpy as np

fixed_coords = np.array([
    [-40.57039, -20.78534],   # sensor 10
    [-40.57175, -20.78040],   # sensor 11
    [-40.57180, -20.78845],   # sensor 12
])

centroid_lon, centroid_lat = fixed_coords.mean(axis=0)
# centroid_lon = -40.57131
# centroid_lat = -20.78473

#Applying the function
wx = fetch_openmeteo_hourly(
    df,
    latitude=centroid_lat,
    longitude=centroid_lon,
    datetime_col="Data-Hora",
)

df= df.merge(wx, left_on="Data-Hora", right_index=True, how="left")

In [ ]:
# fill nan values from Weather Dataframe using xgboosting
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score


def impute_blh_xgboost(
    df: pd.DataFrame,
    datetime_col: str = "Data-Hora",
    target: str = "boundary_layer_height",
    random_state: int = 42,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Imputa NaN em `target` via XGBoost regression usando variáveis meteorológicas
    já presentes no df + features cíclicas de tempo + lags/leads do target.

    Retorna o df com a coluna `target` preenchida e uma coluna nova
    `boundary_layer_height_imputed` (bool) indicando quais linhas foram imputadas.
    """
    df = df.sort_values(datetime_col).reset_index(drop=True).copy()

    # --- Feature engineering ---
    dt = df[datetime_col]
    hour = dt.dt.hour + dt.dt.minute / 60
    doy  = dt.dt.dayofyear

    df["_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["_doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    df["_doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)

    # Radiação acumulada do dia (proxy de aquecimento acumulado → força convectiva)
    df["_swrad_cumsum_day"] = (
        df.groupby(dt.dt.date)["shortwave_radiation"].cumsum()
    )

    # Lags/leads do target (funcionam pq o gap é curto, ~1-2h)
    for k in [1, 2, 3, 24]:
        df[f"_{target}_lag{k}"]  = df[target].shift(k)
        df[f"_{target}_lead{k}"] = df[target].shift(-k)

    # Interação estabilidade: temp - dew_point (quanto maior, mais seco/instável)
    df["_t_minus_td"] = df["temperature_2m"] - df["dew_point_2m"]

    # --- Separar treino (onde BLH existe) e target de imputação (NaN) ---
    feature_cols = [
        "temperature_2m", "relative_humidity_2m", "dew_point_2m",
        "pressure_msl", "surface_pressure",
        "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
        "precipitation", "cloud_cover", "shortwave_radiation",
        "_hour_sin", "_hour_cos", "_doy_sin", "_doy_cos",
        "_swrad_cumsum_day", "_t_minus_td",
        f"_{target}_lag1", f"_{target}_lag2", f"_{target}_lag3", f"_{target}_lag24",
        f"_{target}_lead1", f"_{target}_lead2", f"_{target}_lead3", f"_{target}_lead24",
    ]

    mask_known = df[target].notna()
    mask_impute = df[target].isna()

    X_train = df.loc[mask_known, feature_cols]
    y_train = df.loc[mask_known, target]
    X_pred  = df.loc[mask_impute, feature_cols]

    # --- Validação com 5-fold CV antes de imputar (honestidade metodológica) ---
    model_params = dict(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        tree_method="hist",
        early_stopping_rounds=50,
        random_state=random_state,
    )

    if verbose:
        kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
        rmses, r2s = [], []
        for tr_idx, va_idx in kf.split(X_train):
            m = xgb.XGBRegressor(**model_params)
            m.fit(
                X_train.iloc[tr_idx], y_train.iloc[tr_idx],
                eval_set=[(X_train.iloc[va_idx], y_train.iloc[va_idx])],
                verbose=False,
            )
            pred = m.predict(X_train.iloc[va_idx])
            rmses.append(np.sqrt(mean_squared_error(y_train.iloc[va_idx], pred)))
            r2s.append(r2_score(y_train.iloc[va_idx], pred))
        print(f"[CV 5-fold]  RMSE: {np.mean(rmses):7.1f} ± {np.std(rmses):.1f} m")
        print(f"[CV 5-fold]  R²:   {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
        print(f"[baseline]   σ(y): {y_train.std():7.1f} m   "
              f"(compare com RMSE acima)")

    # --- Modelo final em todo o treino, predição nos NaN ---
    # Holdout de 10% só para permitir early_stopping no fit final
    n = len(X_train)
    rng = np.random.default_rng(random_state)
    val_idx = rng.choice(n, size=n // 10, replace=False)
    tr_mask = np.ones(n, dtype=bool); tr_mask[val_idx] = False

    final_model = xgb.XGBRegressor(**model_params)
    final_model.fit(
        X_train.iloc[tr_mask], y_train.iloc[tr_mask],
        eval_set=[(X_train.iloc[~tr_mask], y_train.iloc[~tr_mask])],
        verbose=False,
    )

    df.loc[mask_impute, target] = final_model.predict(X_pred)
    df[f"{target}_imputed"] = mask_impute.values

    # Limpa colunas auxiliares
    df = df.drop(columns=[c for c in df.columns if c.startswith("_")])

    if verbose:
        print(f"\n[impute]     {mask_impute.sum()} linhas imputadas "
              f"de {len(df)} totais "
              f"({100*mask_impute.mean():.1f}%)")

    return df


# --- Uso ---
df = impute_blh_xgboost(df)

### Generating processed .csv

In [79]:
df

,Data-Hora,Carregamento\n(TMN),Navio\nAtracado,H20 (%),"Finos \n-6,3mm (%)","Compressão\n16,0(Kgf)","Compressão\n12,5(Kgf)",Tempo Estoque\n(dias),Descarga\nPet Coke,Descarga\nCalcário,...,dew_point_2m,pressure_msl,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation,cloud_cover,shortwave_radiation,boundary_layer_height
1765,2024-03-14 12:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,23.4,1015.1,1015.1,4.00,127,9.1,0.0,60,863.0,NaN
1766,2024-03-14 13:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,23.0,1014.6,1014.6,4.02,117,9.5,0.0,98,854.0,NaN
1767,2024-03-14 14:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,22.9,1014.0,1014.0,3.83,105,9.1,0.0,95,702.0,NaN
1768,2024-03-14 15:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,23.4,1013.6,1013.6,3.85,99,8.4,0.0,15,557.0,NaN
1769,2024-03-14 16:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,23.4,1013.3,1013.3,3.41,87,8.3,0.0,25,463.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19404,2026-03-19 11:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24.1,1009.6,1009.6,3.40,91,9.2,0.1,29,816.0,535.0
19405,2026-03-19 12:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24.3,1009.1,1009.1,4.07,53,9.7,0.2,82,741.0,470.0
19406,2026-03-19 13:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23.8,1008.4,1008.4,2.35,1,9.7,0.3,100,551.0,480.0
19407,2026-03-19 14:00:00,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23.9,1007.9,1007.9,1.33,34,6.2,0.6,100,288.0,375.0


In [ ]:
#Save processed dataframe as .csv
output_path = r"/home/jorgemetri/Desktop/git_repositories/Fugitive-Industrial-Emission-Mining-Company/data/processed/Dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /home/jorgemetri/Desktop/git_repositories/Fugitive-Industrial-Emission-Mining-Company/data/processed/Dados Carregamento_24-25-26_RAMPs.csv
